# ChatgaiyyaAlap MT — Day 2 (Google Colab)
Zero-Shot Prompting, Both Directions

**Requires Day 1 to be done first** — this notebook pulls `outputs/sampled_pairs.csv`
from your GitHub repo (pushed at the end of the Day 1 notebook), so make sure that ran
and pushed successfully before starting here.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (optional, faster).


## Step 0 — Install dependencies

In [8]:
!pip -q install transformers accelerate sacrebleu pandas tqdm


## Step 2 — Load the Day 1 sample

This must be the exact same `outputs/sampled_pairs.csv` your whole team is using —
don't regenerate it here.


In [9]:
import pandas as pd
from pathlib import Path

SAMPLE_PATH = Path("/content/sampled_pairs.csv")
if not SAMPLE_PATH.exists():
    raise FileNotFoundError(
        "outputs/sampled_pairs.csv not found. Run the Day 1 notebook first and make "
        "sure it pushed outputs/ to GitHub, or manually copy it into this repo."
    )

sample = pd.read_csv(SAMPLE_PATH)
print(f"Loaded {len(sample)} sentence pairs for Day 2.")
sample.head()


Loaded 60 sentence pairs for Day 2.


,bangla,chatgaiya,bangla_len,chatgaiya_len
0,সব সময় এমন জুতা দিয়ে মাইর খায়,অক্কল সমত এন জুতা দি মাইর হায়,7,7
1,অনেক দূরে যাবার পর অন্য দিকে ঘুরে গেল,বহুত দূরে যায়বের ফর অন্য মিক্কে ঘুরি গিল,8,8
2,বউকে ধরে ঘুষি দিতে হবে,বউরে ধরি কিলদন অইব্যু,5,4
3,কোথায় যাবো তোমার মেয়েকে নিয়ে,হডে যাইয়্যুম তোঁয়ার মাইয়্যারে লইয়্যে,5,5
4,ও এখন কিছু করে না,ইতে এহন কিসু ন গরে,5,5


## Step 3 — Write the zero-shot prompt templates

One template per direction. Zero-shot = instruction only, no translated examples.
Kept as plain `.txt` files under `prompts/` per the team's repo structure, with `{text}`
as the substitution point, so they're easy to diff/version and reuse in Day 3/4.

**Design notes (log these in your own words in your learning notes):**
- Explicit source/target language names reduce ambiguity for the model.
- "Reply with only the translation" curbs the model's tendency to explain itself or
  add commentary, which would otherwise pollute BLEU/chrF scoring.
- Chatgaiya has no standardized spelling — watch for the model defaulting back to
  Standard Bangla instead of actually attempting the dialect. That's a documented
  zero-shot failure mode, log it in your issue log if you see it.


In [10]:
import os
os.makedirs("prompts", exist_ok=True)

zero_shot_b2c = (
    "Translate the following sentence from Standard Bangla into the Chittagonian "
    "(Chatgaiya) dialect spoken in southeastern Bangladesh. Chatgaiya is a distinct "
    "regional dialect, not Standard Bangla. Reply with only the translated sentence "
    "in Chatgaiya, and nothing else.\n\n"
    "Standard Bangla: {text}\n"
    "Chittagonian:"
)

zero_shot_c2b = (
    "Translate the following sentence from the Chittagonian (Chatgaiya) dialect of "
    "southeastern Bangladesh into Standard Bangla. Reply with only the translated "
    "sentence in Standard Bangla, and nothing else.\n\n"
    "Chittagonian: {text}\n"
    "Standard Bangla:"
)

with open("prompts/zero_shot_b2c_v1.txt", "w", encoding="utf-8") as f:
    f.write(zero_shot_b2c)
with open("prompts/zero_shot_c2b_v1.txt", "w", encoding="utf-8") as f:
    f.write(zero_shot_c2b)

print(zero_shot_b2c)
print("---")
print(zero_shot_c2b)


Translate the following sentence from Standard Bangla into the Chittagonian (Chatgaiya) dialect spoken in southeastern Bangladesh. Chatgaiya is a distinct regional dialect, not Standard Bangla. Reply with only the translated sentence in Chatgaiya, and nothing else.

Standard Bangla: {text}
Chittagonian:
---
Translate the following sentence from the Chittagonian (Chatgaiya) dialect of southeastern Bangladesh into Standard Bangla. Reply with only the translated sentence in Standard Bangla, and nothing else.

Chittagonian: {text}
Standard Bangla:


## Step 4 — Load the model

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # keep consistent with Day 1 unless your team agreed to change it

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()
print(f"Loaded on device: {device}")

GEN_CONFIG = dict(max_new_tokens=100, temperature=0.3, do_sample=True)
print("Generation config (log these values — you'll be asked about them):", GEN_CONFIG)


Loading Qwen/Qwen2.5-1.5B-Instruct ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded on device: cuda
Generation config (log these values — you'll be asked about them): {'max_new_tokens': 100, 'temperature': 0.3, 'do_sample': True}


## Step 5 — Run zero-shot translation across the full sample, both directions

In [12]:
import time
from tqdm.auto import tqdm

def translate(text, template):
    prompt_text = template.format(text=text)
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, **GEN_CONFIG)
    latency = time.time() - start

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    text_out = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return text_out, latency


def run_direction(df, template, source_col, ref_col):
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        prediction, latency = translate(row[source_col], template)
        rows.append({
            "source": row[source_col],
            "reference": row[ref_col],
            "prediction": prediction,
            "latency_sec": round(latency, 3),
        })
    return rows


print("Running zero-shot: Bangla -> Chatgaiya ...")
results_b2c = run_direction(sample, zero_shot_b2c, source_col="bangla", ref_col="chatgaiya")

print("Running zero-shot: Chatgaiya -> Bangla ...")
results_c2b = run_direction(sample, zero_shot_c2b, source_col="chatgaiya", ref_col="bangla")


Running zero-shot: Bangla -> Chatgaiya ...


  0%|          | 0/60 [00:00<?, ?it/s]

Running zero-shot: Chatgaiya -> Bangla ...


  0%|          | 0/60 [00:00<?, ?it/s]

## Step 6 — Score each example with BLEU and chrF

This gives immediate per-example signal for spotting hallucinations/failures now.
The *formal*, aggregated evaluation (including repeated-run consistency checks across
zero-shot, few-shot, and dictionary-variant together) happens Day 5 in `day5_eval.json`
— this is just enough to sanity-check Day 2 output quality and flag issues early.


In [13]:
import sacrebleu

def score_pair(prediction, reference):
    if not prediction.strip():
        return 0.0, 0.0
    bleu = sacrebleu.sentence_bleu(prediction, [reference]).score / 100
    chrf = sacrebleu.sentence_chrf(prediction, [reference]).score / 100
    return round(bleu, 4), round(chrf, 4)


def add_scores(rows):
    for r in rows:
        bleu, chrf = score_pair(r["prediction"], r["reference"])
        r["bleu"] = bleu
        r["chrf"] = chrf
    return rows


results_b2c = add_scores(results_b2c)
results_c2b = add_scores(results_c2b)

import statistics

def avg(rows, key):
    return round(statistics.mean(r[key] for r in rows), 4)

print("Bangla -> Chatgaiya  avg BLEU:", avg(results_b2c, "bleu"), " avg chrF:", avg(results_b2c, "chrf"))
print("Chatgaiya -> Bangla  avg BLEU:", avg(results_c2b, "bleu"), " avg chrF:", avg(results_c2b, "chrf"))


Bangla -> Chatgaiya  avg BLEU: 0.0415  avg chrF: 0.1389
Chatgaiya -> Bangla  avg BLEU: 0.0469  avg chrF: 0.1598


## Step 7 — Spot-check a few examples before saving

Actually look at a handful, including any that scored near zero — that's where
hallucinations or refusals usually show up. Log anything odd in `logs/issue_log.txt`.


In [14]:
import random

print("=== Bangla -> Chatgaiya, 3 random examples ===")
for r in random.sample(results_b2c, 3):
    print(f"SRC:  {r['source']}")
    print(f"REF:  {r['reference']}")
    print(f"PRED: {r['prediction']}")
    print(f"BLEU: {r['bleu']}  chrF: {r['chrf']}")
    print("-" * 60)

print("\n=== Worst-scoring examples (chrF) — check these for hallucinations ===")
worst = sorted(results_b2c + results_c2b, key=lambda r: r["chrf"])[:5]
for r in worst:
    print(f"SRC:  {r['source']}")
    print(f"REF:  {r['reference']}")
    print(f"PRED: {r['prediction']}")
    print(f"chrF: {r['chrf']}")
    print("-" * 60)


=== Bangla -> Chatgaiya, 3 random examples ===
SRC:  তোকে বিয়ে করতে হবে না
REF:  তুঁরে বিয়া গরিতে ন অইব্যু
PRED: তোকে স্বাগতম করতে হবেন
BLEU: 0.0  chrF: 0.1145
------------------------------------------------------------
SRC:  আমি চাচার সাথে একটু খাতির করার চেষ্টা করি
REF:  অ্যাঁই চাচার লগে এক্কানা খাতির গরিবের চেষ্টা গরি
PRED: আমি চাচার পাই বালিয়া গেছি।
BLEU: 0.0586  chrF: 0.132
------------------------------------------------------------
SRC:  আমি অহংকার করি আমি চট্টগ্রামের ছেলে আমার জন্ম এখানে আমার অহংকার
REF:  অ্যাঁই দেমাগ গরি অ্যাঁই চিটাংঅর ফোয়া অ্যাঁর জন্ম ইয়েন অ্যাঁর দেমাগ
PRED: আমি হোয়াইন খালি আমি তিনি চট্টগ্রামের বড় বড় ছেলে আমার জন্ম এখানে আমার হোয়াইন
BLEU: 0.0284  chrF: 0.1304
------------------------------------------------------------

=== Worst-scoring examples (chrF) — check these for hallucinations ===
SRC:  রাজা হইলো তা লইয়্যে
REF:  রাজা বলল তা নিয়ে
PRED: The king was there.
chrF: 0.0
------------------------------------------------------------
SRC:  দুরাদুরি শুরু

In [17]:
# Append anything notable to the issue log — edit this list with what you actually observed
os.makedirs("logs", exist_ok=True)

issue_notes = [
     "b2c example 12: output was in English instead of Chatgaiya",
     "c2b: model frequently just echoed the Chatgaiya input unchanged (zero-shot didn't understand the task)",
]

with open("logs/issue_log.txt", "a", encoding="utf-8") as f:
    f.write(f"\n--- Day 2 zero-shot ({pd.Timestamp.utcnow()}) ---\n")
    if issue_notes:
        for note in issue_notes:
            f.write(f"- {note}\n")
    else:
        f.write("- (fill in real observations before submitting — don't leave this empty)\n")

print("Logged to logs/issue_log.txt — go edit issue_notes above with your real findings.")


Logged to logs/issue_log.txt — go edit issue_notes above with your real findings.


## Step 8 — Save as `day2_zeroshot_b2c.json` and `day2_zeroshot_c2b.json`

In [16]:
import json
from datetime import datetime, timezone

def build_output(direction, template_file, results):
    return {
        "dataset": "ChatgaiyyaAlap",
        "direction": direction,
        "technique": "zero_shot",
        "model": MODEL_NAME,
        "prompt_template": template_file,
        "generation_config": GEN_CONFIG,
        "num_examples": len(results),
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": results,
        "metrics": {
            "avg_bleu": avg(results, "bleu"),
            "avg_chrf": avg(results, "chrf"),
            "avg_latency_sec": round(statistics.mean(r["latency_sec"] for r in results), 3),
        },
    }

output_b2c = build_output("bangla_to_chatgaiya", "prompts/zero_shot_b2c_v1.txt", results_b2c)
output_c2b = build_output("chatgaiya_to_bangla", "prompts/zero_shot_c2b_v1.txt", results_c2b)

os.makedirs("outputs", exist_ok=True)
with open("outputs/day2_zeroshot_b2c.json", "w", encoding="utf-8") as f:
    json.dump(output_b2c, f, ensure_ascii=False, indent=2)
with open("outputs/day2_zeroshot_c2b.json", "w", encoding="utf-8") as f:
    json.dump(output_c2b, f, ensure_ascii=False, indent=2)

print("Saved outputs/day2_zeroshot_b2c.json")
print("Saved outputs/day2_zeroshot_c2b.json")


Saved outputs/day2_zeroshot_b2c.json
Saved outputs/day2_zeroshot_c2b.json


---
**Recap — Day 2 checklist:**
1. ✅ Zero-shot prompt template, Bangla → Chatgaiya
2. ✅ Zero-shot prompt template, Chatgaiya → Bangla
3. ✅ Ran both across the full Day 1 sample
4. ✅ Stored raw outputs alongside references
5. ✅ Saved `day2_zeroshot_b2c.json` and `day2_zeroshot_c2b.json`

**Before your next session, be ready to answer:**
- Which direction did the model handle worse, and what pattern do you see in the failures?
- Did it ever default back to Standard Bangla instead of attempting Chatgaiya?
- Any hallucinated words that *look* like Chatgaiya but aren't?

Next: Day 3 — few-shot prompting, same sample, same two directions.
